# SLM vs flagship LLMs on financial report QA (TAT-QA)

Everything lives in the GitHub repo. This notebook only **pulls the repo, loads your Kaggle Secrets, and runs the pipeline**.
To change anything (settings, models, prices, which steps run), edit `kaggle/config.toml` on GitHub and re-run. Never edit these cells.

**One-time setup**
- Settings: **Internet = On**
- Add-ons -> Secrets (tick each for this notebook): `HF_TOKEN` (write token, required for checkpoint backup),
  `LLM_API_KEY` / `ANTHROPIC_API_KEY` (only for the API comparison). `GITHUB_TOKEN` only if the repo is private.

**How to run**
1. GPU run - Accelerator **GPU T4 x2**, `STAGES = ""`, then **Save Version -> Save & Run All (Commit)**.
   Trains and tests the local models; if the session stops, commit again and it resumes from the last checkpoint.
2. API comparison - after the GPU run, Accelerator **None** (CPU), `STAGES = "setup,data,api,evaluate"`, commit again.


In [ ]:
# Cell 1 - get the latest code from GitHub and load secrets. This cell never needs to change.
REPO_URL = "https://github.com/niksalunke/FINETUNINGSLM.git"
SECRETS = ["GITHUB_TOKEN", "HF_TOKEN", "LLM_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY"]

import base64, os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    _client = UserSecretsClient()
    for _name in SECRETS:
        try:
            os.environ[_name] = _client.get_secret(_name).strip()
            print("secret loaded:", _name)
        except Exception:
            pass
except ImportError:
    print("not running on Kaggle - using environment variables")

_token = os.environ.get("GITHUB_TOKEN", "")
_auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""
_git = ["git"] + (["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if _token else [])

os.chdir("/kaggle/working")
if os.path.isdir("FINETUNINGSLM/.git"):
    _r = subprocess.run(_git + ["-C", "FINETUNINGSLM", "pull", "--quiet", "--ff-only"], capture_output=True, text=True)
else:
    subprocess.run(["rm", "-rf", "FINETUNINGSLM"])
    _r = subprocess.run(_git + ["clone", "--quiet", REPO_URL, "FINETUNINGSLM"], capture_output=True, text=True)
if _r.returncode != 0:
    _msg = _r.stderr
    for _s in (_token, _auth):
        if _s:
            _msg = _msg.replace(_s, "***")
    raise SystemExit("git failed (check the GITHUB_TOKEN secret is attached and can read the repo):" + chr(10) + _msg)

os.chdir("/kaggle/working/FINETUNINGSLM")
print("code version:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Cell 2 - run the pipeline.
#   GPU run (Accelerator GPU T4 x2):             STAGES = ""   (stages listed in kaggle/config.toml)
#   API comparison afterwards (Accelerator None): STAGES = "setup,data,api,evaluate"
STAGES = ""

_flag = f"--stages {STAGES}" if STAGES else ""
!python kaggle/run_pipeline.py {_flag}